# TeleRAG-Agent: Gradio UI on Kaggle

This notebook launches the full TeleRAG-Agent Gradio interface on a Kaggle T4 GPU.

### Before running:
1. Set **Accelerator** → **GPU T4 x2** in notebook settings.
2. Toggle **Internet** → **ON** in notebook settings.
3. Click **Add Input** (right panel) → **Datasets** → find your `telerag-qdrant-db` dataset → click **+**.
4. Update `QDRANT_DATASET_PATH` below to match the actual path shown in `/kaggle/input/`.
5. Run the single cell below. A `gradio.live` public link will appear at the bottom.

In [ ]:
import os, sys

# ================================================================
# CONFIGURATION — UPDATE THIS ONE LINE
# ================================================================
# Run `!ls /kaggle/input/` first if you are unsure of the name.
QDRANT_DATASET_PATH = "/kaggle/input/telerag-qdrant-db/qdrant_storage"
# ================================================================

# ── STEP 1: Fix sentence-transformers BEFORE pip install -r requirements ──
# Kaggle pre-installs a very new sentence-transformers (>=3.x) which pulls
# in torchcodec -> needs FFmpeg shared libs -> NOT available -> crashes.
# Downgrade to 2.7.0 which is the last version without this dependency.
print("Step 1/5: Fixing sentence-transformers version (avoids torchcodec/FFmpeg crash)...")
os.system("pip install sentence-transformers==2.7.0 -q --no-deps")
os.system("pip install sentence-transformers==2.7.0 -q")

# ── STEP 2: Clone repository ──────────────────────────────────────────────
print("\nStep 2/5: Cloning TeleRAG-Agent repository...")
if not os.path.exists("/kaggle/working/TeleRAG-Agent"):
    os.system("git clone https://github.com/imchaitanya0/TeleRAG-Agent.git /kaggle/working/TeleRAG-Agent")
else:
    print("  Repo already exists, pulling latest...")
    os.system("cd /kaggle/working/TeleRAG-Agent && git pull")

os.chdir("/kaggle/working/TeleRAG-Agent")
sys.path.insert(0, "/kaggle/working/TeleRAG-Agent")

# ── STEP 3: Install remaining dependencies ────────────────────────────────
print("\nStep 3/5: Installing remaining dependencies from requirements.txt...")
# Exclude vllm (incompatible with Kaggle's PyTorch build) and dev tools
os.system("""
pip install \
    langgraph>=0.2.0 langchain-core>=0.2.0 \
    qdrant-client>=1.9.0 fastembed>=0.3.6 \
    FlagEmbedding>=1.2.0 \
    gradio>=4.0.0 \
    python-dotenv pyyaml networkx tqdm \
    -q
""")

# ── STEP 4: Set up Qdrant database folder ────────────────────────────────
print("\nStep 4/5: Setting up local Qdrant database...")
os.makedirs("data", exist_ok=True)
dest = "data/qdrant_storage"
if os.path.exists(QDRANT_DATASET_PATH):
    if not os.path.exists(dest):
        print(f"  Copying Qdrant storage from {QDRANT_DATASET_PATH} ...")
        os.system(f"cp -r {QDRANT_DATASET_PATH} {dest}")
        print("  Done.")
    else:
        print(f"  Qdrant storage already in place at {dest}.")
else:
    print(f"  ⚠️  WARNING: Qdrant storage not found at {QDRANT_DATASET_PATH}")
    print("  Check: !ls /kaggle/input/ to see available datasets.")
    print("  The Chat tab will start but retrieval will fail.")

# ── STEP 5: Launch Gradio UI ─────────────────────────────────────────────
print("\nStep 5/5: Launching Gradio UI...")
print("NOTE: The LLM (16GB) loads into GPU first — expect 2-3 min before UI responds.")
print("Look for the 'gradio.live' public link printed below.\n")

# This blocks until you stop the kernel — that is expected!
os.system("python src/ui/app.py --share")


## Troubleshooting

**Check what datasets are mounted:**
```python
!ls /kaggle/input/
!ls /kaggle/input/telerag-qdrant-db/   # adjust name as needed
```

**Verify sentence-transformers version (should be 2.7.0):**
```python
import sentence_transformers; print(sentence_transformers.__version__)
```

**Verify CUDA is available:**
```python
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
```